# Managed Agents: the agentic loop and sessions

> **CCA-F Domain 1 - Agentic (27%).** The exam tests that you stop the loop on **`session.status_idle`** and read **`stop_reason`**, not on vibes. It also tests that a **session** carries memory server-side across turns, so you do not resend prior turns.

Theme: a **70s slasher plus giallo double-feature curator**. We stand up a **Managed Agent**, run one turn to idle, then run a second turn on the **same session** to prove the server remembers the first film without us resending it.

### 1. Install dependencies

In [ ]:
# uv-managed venvs ship WITHOUT pip, so `%pip install` fails here.
# Use uv instead, pointed at THIS kernel's interpreter so packages land
# where the kernel actually looks. Idempotent: uv no-ops if already satisfied.
import subprocess, sys

subprocess.run(
    ["uv", "pip", "install", "--python", sys.executable, "anthropic", "python-dotenv"],
    check=True,
)
print("Dependencies ready.")

### 2. Load environment variables from .env file

In [ ]:
from dotenv import load_dotenv
load_dotenv()

### 3. Create an API client

The Managed Agents surface is **beta-gated**. We set `BETA` to the beta header value once and pass `betas=[BETA]` on every call. `MODEL` stays on the repo default **Haiku 4.5**.

In [ ]:
from anthropic import Anthropic

client = Anthropic()
MODEL = "claude-haiku-4-5"          # repo default; Sonnet only where reasoning depth earns it
BETA = "managed-agents-2026-04-01"  # beta header for the whole Managed Agents surface

### 4. Create the agent, the environment, and the session

Three server-side resources, three verbs. **`agents.create`** takes `model={"id": MODEL}` plus a `name`, with `system` optional. **`environments.create`** is the scratch space the runtime executes in. **`sessions.create`** binds an agent to an environment - note `agent=` takes the **agent ID string** and `environment_id=` is required. The **session** is the runtime that holds conversation state.

In [ ]:
# 1. Create the curator agent. The system prompt sets the double-feature persona.
agent = client.beta.agents.create(
    model={"id": MODEL},
    name="double-feature-curator",
    system=(
        "You are a 70s slasher and giallo double-feature curator. "
        "When asked for a pairing, name two films and give one sharp sentence on why they rhyme "
        "(kills, color, score, or director). Keep it tight."
    ),
    betas=[BETA],
)

# 2. Create the environment the runtime executes in.
env = client.beta.environments.create(name="curator-scratch", betas=[BETA])

# 3. Create the session that binds this agent to this environment.
session = client.beta.sessions.create(
    agent=agent.id,               # the agent ID string, not the object
    environment_id=env.id,
    betas=[BETA],
)

print("agent:  ", agent.id)
print("env:    ", env.id)
print("session:", session.id)

### 5. Send the first turn and run the loop until idle

We **`send`** a `user.message` event, then **`stream`** events back. The agentic loop is done when the stream yields **`session.status_idle`** - that event carries **`stop_reason`**. We collect `agent.message` text as it arrives and break on idle.

> **Gotcha.** Do not stop the loop when the assistant text "looks finished." Stop on **`session.status_idle`**. Mid-turn the stream can emit `agent.tool_use` events that the runtime still has to resolve before the turn ends; breaking early cuts the turn off before `stop_reason` is set.

In [ ]:
def run_turn(text: str) -> str:
    """Send one user turn, drive the loop to idle, and return the agent's reply.

    The teaching point: the loop terminates on `session.status_idle`, whose
    `stop_reason` is the authoritative end signal - not the presence of text.
    """
    client.beta.sessions.events.send(
        session.id,
        events=[{"type": "user.message",
                 "content": [{"type": "text", "text": text}]}],
        betas=[BETA],
    )

    reply_parts: list[str] = []
    for event in client.beta.sessions.events.stream(session.id, betas=[BETA]):
        if event.type == "agent.message":
            # Accumulate assistant text as the runtime streams it.
            for block in event.content:
                if getattr(block, "type", None) == "text":
                    reply_parts.append(block.text)
        elif event.type == "session.status_idle":
            # The ONLY correct place to stop. stop_reason lives on this event.
            print("stop_reason:", event.stop_reason)
            break

    return "".join(reply_parts)


first_reply = run_turn("Pair a Bava giallo with a Carpenter slasher for tonight.")
print(first_reply)

### 6. Second turn on the same session (server-side memory)

We ask a follow-up that only makes sense if the agent still remembers the first pairing. We do **not** resend the earlier films. The **session holds conversation state server-side**, so the second `send` on the same `session.id` continues the thread. This is the Domain 1 point: with Managed Agents you carry a **session ID**, not a growing local transcript.

In [ ]:
# No films named here. If the agent answers correctly, the SERVER remembered
# the first turn - proof the session carries memory across turns.
second_reply = run_turn("Which of those two should we watch FIRST, and why?")
print(second_reply)

### 7. Tear down

Always **archive** what you created so no billable session dangles. There is **no `agents.delete`**; the verb is **`archive`**. Archive the session first, then the agent.

In [ ]:
client.beta.sessions.archive(session.id, betas=[BETA])
client.beta.agents.archive(agent.id, betas=[BETA])
print("Archived session and agent. Clean exit.")